| Project                                | Libraries                                        | Topics Covered                                                  |
| -------------------------------------- | ------------------------------------------------ | --------------------------------------------------------------- |
| **3. Spam Email Classifier**           | Scikit-learn, NLTK                               | TF-IDF, CountVectorizer, Naive Bayes, Logistic Regression, SVM  |

In [111]:
# Importing Libraries
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import contractions
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.linear_model import  LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SAI\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SAI\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SAI\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [112]:
# Loading our dataset
df = pd.read_csv(r'D:\HIMANSHU\Desktop\RD\UDEMY CODE\NLP PROJECTS +++\3. Spam Email Classifier\email_spam.csv', encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives around here though",NaN,NaN,NaN


In [113]:
# Data Cleaning 
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)

df = df.rename(columns={'v1':'Category', 'v2':'Text'})

df['Category'] = df['Category'].map({'ham':0, 'spam':1})

df.head()

,Category,Text
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives around here though"


In [114]:
# EDA
df['Text'].describe()

df['count_len'] = df['Text'].apply(len)

df['Category'].value_counts(normalize=True)*100

pd.set_option('display.max_colwidth', None)
df.head()

,Category,Text,count_len
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",111
1,0,Ok lar... Joking wif u oni...,29
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,155
3,0,U dun say so early hor... U c already then say...,49
4,0,"Nah I don't think he goes to usf, he lives around here though",61


In [115]:
# Data Preprocessing 
df['Text'] = df['Text'].map(lambda s : s.lower() if isinstance(s,str) else s)             # lowercasing

df['Text'] = df['Text'].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True)         # removed numbers and special characters

df['Text'] = df['Text'].apply(contractions.fix)

df.head()

,Category,Text,count_len
0,0,go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat,111
1,0,ok lar joking wif you oni,29
2,1,free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005 text fa to 87121 to receive entry questionstd txt ratetcs apply 08452810075over18s,155
3,0,you dun say so early hor you c already then say,49
4,0,nah i do not think he goes to usf he lives around here though,61


In [116]:
# Word Tokenize and Stopwords removal
# txt = 'Hello i am ai'
# word_tokenize(txt)

tokens = df['Text'].map(word_tokenize)

stop_words = set(stopwords.words('english'))
df['Text'] = tokens.map(lambda tokens : [word for word in tokens if word not in stop_words])
df.head()


,Category,Text,count_len
0,0,"[go, jurong, point, crazy, available, bugis, n, great, world, la, e, buffet, cine, got, amore, wat]",111
1,0,"[ok, lar, joking, wif, oni]",29
2,1,"[free, entry, 2, wkly, comp, win, fa, cup, final, tkts, 21st, may, 2005, text, fa, 87121, receive, entry, questionstd, txt, ratetcs, apply, 08452810075over18s]",155
3,0,"[dun, say, early, hor, c, already, say]",49
4,0,"[nah, think, goes, usf, lives, around, though]",61


In [117]:
# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df.Text, df.Category, test_size=0.25)
X_train = X_train.map(" ". join)
X_test =  X_test.map(" ". join)
# X_train[2]

**TF - IDF Vectorizer**

In [118]:
# TF - IDF Vectorizer
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [119]:
nb = BernoulliNB()
nb.fit(X_train_tfidf, y_train)

nb_pred = nb.predict(X_test_tfidf)

print(f"\n========== Naive Bayes Classifier ==========\n")
print(f"classification Report : \n {classification_report(nb_pred, y_test)}")

# ==============================

lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

lr_pred = lr.predict(X_test_tfidf)

print(f"\n========== Logistic Regression ==========\n")
print(f"classification Report : \n {classification_report(lr_pred, y_test)}")

# ==============================

svm = SVC()
svm.fit(X_train_tfidf, y_train)

svm_pred = svm.predict(X_test_tfidf)

print(f"\n========== Support Vector Classifier ==========\n")
print(f"classification Report : \n {classification_report(svm_pred, y_test)}")

# ==============================

rf = RandomForestClassifier()
rf.fit(X_train_tfidf, y_train)

rf_pred = rf.predict(X_test_tfidf)

print(f"\n========== Random Forest Classifier ==========\n")
print(f"classification Report : \n {classification_report(rf_pred, y_test)}")


========== Naive Bayes Classifier ==========

classification Report : 
               precision    recall  f1-score   support

           0       1.00      0.96      0.98      1251
           1       0.75      1.00      0.86       142

    accuracy                           0.97      1393
   macro avg       0.87      0.98      0.92      1393
weighted avg       0.97      0.97      0.97      1393


========== Logistic Regression ==========

classification Report : 
               precision    recall  f1-score   support

           0       1.00      0.96      0.98      1256
           1       0.72      1.00      0.84       137

    accuracy                           0.96      1393
   macro avg       0.86      0.98      0.91      1393
weighted avg       0.97      0.96      0.96      1393


========== Support Vector Classifier ==========

classification Report : 
               precision    recall  f1-score   support

           0       1.00      0.97      0.99      1237
           1      

In [120]:
# Testing on new unseen data using TFIDF Vectorizer
import re

text = "free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply overs"

def preprocess(text):
    lowercase_text = text.lower()
    special_char = re.sub(r"[^a-zA-Z\s]", "", lowercase_text)
    contraction_text = contractions.fix(special_char)
    tokenized_text = word_tokenize(contraction_text)
    stopwords_text = [word for word in tokenized_text if word not in stop_words]
    final_text = " ".join(stopwords_text)

    return final_text

text_tfidf = tfidf.transform([preprocess(text)])

text_pred = svm.predict(text_tfidf)
text_pred

array([1])

**Count vectorizer**

In [121]:
# Count Vectorizer
cv = CountVectorizer()

X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

In [122]:
nb = BernoulliNB()
nb.fit(X_train_cv, y_train)

nb_pred = nb.predict(X_test_cv)

print(f"\n========== Naive Bayes Classifier ==========\n")
print(f"classification Report : \n {classification_report(nb_pred, y_test)}")

# ==============================

lr = LogisticRegression()
lr.fit(X_train_cv, y_train)

lr_pred = lr.predict(X_test_cv)

print(f"\n========== Logistic Regression ==========\n")
print(f"classification Report : \n {classification_report(lr_pred, y_test)}")

# ==============================

svm = SVC()
svm.fit(X_train_cv, y_train)

svm_pred = svm.predict(X_test_cv)

print(f"\n========== Support Vector Classifier ==========\n")
print(f"classification Report : \n {classification_report(svm_pred, y_test)}")

# ==============================

rf = RandomForestClassifier()
rf.fit(X_train_cv, y_train)

rf_pred = rf.predict(X_test_cv)

print(f"\n========== Random Forest Classifier ==========\n")
print(f"classification Report : \n {classification_report(rf_pred, y_test)}")


========== Naive Bayes Classifier ==========

classification Report : 
               precision    recall  f1-score   support

           0       1.00      0.96      0.98      1251
           1       0.75      1.00      0.86       142

    accuracy                           0.97      1393
   macro avg       0.87      0.98      0.92      1393
weighted avg       0.97      0.97      0.97      1393


========== Logistic Regression ==========

classification Report : 
               precision    recall  f1-score   support

           0       1.00      0.98      0.99      1231
           1       0.85      1.00      0.92       162

    accuracy                           0.98      1393
   macro avg       0.93      0.99      0.95      1393
weighted avg       0.98      0.98      0.98      1393


========== Support Vector Classifier ==========

classification Report : 
               precision    recall  f1-score   support

           0       1.00      0.97      0.99      1235
           1      

In [123]:
# Testing on new unseen data using Count Vectorizer
import re

text = "Congratulations! You have won a FREE iPhone. Click the link below to claim your prize now."

def preprocess(text):
    lowercase_text = text.lower()
    special_char = re.sub(r"[^a-zA-Z\s]", "", lowercase_text)
    contraction_text = contractions.fix(special_char)
    tokenized_text = word_tokenize(contraction_text)
    stopwords_text = [word for word in tokenized_text if word not in stop_words]
    final_text = " ".join(stopwords_text)

    return final_text

text_cv = cv.transform([preprocess(text)])

text_pred = svm.predict(text_cv)
text_pred

array([1])